# 2 — XGBoost (Tuned Gradient-Boosting Baseline)

This notebook trains and evaluates XGBoost, which serves as the baseline model in this study. XGBoost is the industry-standard approach for structured tabular prediction tasks and represents the traditional gradient-boosting paradigm.

XGBoost works by building decision trees sequentially, where each new tree is designed to correct the errors made by all previous trees. The predictions of all trees are combined to produce the final output.

## Setup

Clone the repository (if running in Colab) and import the project modules.

In [ ]:
import os, sys

if not os.path.isdir("src"):
    !git clone https://github.com/andreasz24/Thesis-Artifact.git
    %cd Thesis-Artifact
    !pip install -r requirements.txt --quiet

sys.path.insert(0, os.getcwd())
print("Ready! Working directory:", os.getcwd())

## Load the train/test split

The preprocessed data split is loaded from `outputs/splits.pkl`. If it doesn't exist yet (e.g. if notebook 1 hasn't been run in this session), it is created automatically using the same deterministic pipeline (`random_state=42`).

In [ ]:
import os, pandas as pd
from src import data, models, evaluation, plotting, config

# If splits.pkl doesn't exist yet, create it automatically
if not os.path.exists(config.SPLITS_PATH):
    print("splits.pkl not found — running preprocessing first...")
    X_train, X_test, y_train, y_test = data.build_splits()
else:
    X_train, X_test, y_train, y_test = data.load_splits()

print('Train:', X_train.shape, '| Test:', X_test.shape)

## XGBoost configuration

The model is configured with the following hyperparameters:

- **`n_estimators = 500`** — the number of trees built. Each tree corrects the errors of all previous trees. 500 provides sufficient rounds of error correction without excessive training time.
- **`max_depth = 6`** — how deep each individual tree can grow (up to 6 sequential splits). Deeper trees are more expressive but more prone to overfitting.
- **`learning_rate = 0.05`** — shrinks each new tree's contribution by a factor of 0.05, making the model approach the solution gradually rather than aggressively. A low learning rate is paired with a high number of trees.
- **`subsample = 0.8`** — each tree is trained on a random 80% of the training rows, reducing overfitting to specific observations.
- **`colsample_bytree = 0.8`** — each tree uses a random 80% of the features, reducing overfitting to specific features.

These parameters are defined in `src/models.py` in the `train_xgboost()` function.

## Training

The cell below trains the XGBoost model on the training set. The test set is passed as an evaluation set so that XGBoost can track the validation loss during training (used internally for early stopping diagnostics, though we don't apply early stopping here).

In [ ]:
model, train_time = models.train_xgboost(X_train, y_train, X_test, y_test)
print(f'Training time: {train_time:.2f}s')

## Evaluation

We evaluate the model using three metrics:

- **Accuracy** — proportion of correctly classified observations. Simple but can be misleading with imbalanced classes (a model that always predicts BBB would achieve decent accuracy just because BBB is the most common class).
- **F1 Macro** — computes F1 for each class independently and averages them equally, regardless of class size. This penalises the model for poor performance on rare classes like C, CC, and AAA.
- **F1 Weighted** — averages F1 across classes weighted by their frequency, giving a more representative picture of overall performance.

In [ ]:
results, y_pred, infer_time = models.evaluate_and_save_xgboost(
    model, X_test, y_test, train_time)
evaluation.print_metrics('XGBoost', results, train_time, infer_time, len(X_test))

## Per-class report

The per-class breakdown shows precision, recall, and F1 for each rating category. This reveals where the model succeeds and where it fails.

The pattern is consistent with the class imbalance: the model performs best on the most populated classes (B, BBB, A) which dominate the training set. Performance drops sharply at the extremes — C, CC, and AAA score zero across all metrics because the model never predicts them due to their extremely small representation in the training data.

AA is an interesting case: despite having 112 observations in the test set (significantly more than the rarest classes), it still achieves very poor scores. This suggests the model struggles to distinguish AA from its neighbouring classes (A and AAA), likely because their financial ratio profiles overlap significantly.

In [ ]:
print(evaluation.per_class_report(y_test, y_pred))

## Confusion matrix

The confusion matrix shows the full picture of what the model predicted vs the true rating. The left matrix shows raw counts, the right shows row-normalised proportions (each row sums to 1.0).

Key patterns to look for:

- **Diagonal dominance** — correct predictions appear on the diagonal. Stronger diagonal = better model.
- **Off-diagonal spread** — errors concentrated near the diagonal (adjacent classes) are less severe than errors far from the diagonal.
- **Modal bias** — XGBoost shows a systematic pull toward BBB (the most common class). Companies rated below BBB tend to be over-predicted (pushed up toward BBB), while companies rated above BBB tend to be under-predicted (pulled down toward BBB). This is a well-known consequence of class imbalance.

In [ ]:
plotting.plot_confusion_pair(y_test, y_pred, 'XGBoost', cmap='Blues',
    save_path=config.OUTPUTS_DIR/'xgb_confusion_matrix.png')

## Feature importance

XGBoost provides built-in feature importance scores based on how frequently each feature is used in tree splits and how much it contributes to reducing the loss.

The chart below shows the top 15 most important features. In the thesis results, Pre-Tax Profit Margin and Long-term Debt / Capital ranked highest, reflecting the established view that profitability and leverage are the primary drivers of creditworthiness.

Note: this importance method is specific to tree-based models. TabPFN and AutoGluon use permutation importance instead (a model-agnostic method), so the rankings are not directly comparable across models — though the overall patterns are similar.

In [ ]:
feat_imp = pd.Series(model.feature_importances_, index=X_train.columns)
plotting.plot_feature_importance(feat_imp, 'XGBoost', top_n=15,
    save_path=config.OUTPUTS_DIR/'xgb_feature_importance.png')

## Summary

XGBoost establishes the baseline performance for this study. The results and model are saved to `outputs/xgb_results.pkl` for later comparison with TabPFN-3 (notebook 3) and AutoGluon (notebook 4).